# Agent hello world — ask first, inspect second

This is the first live exercise. We will ask Stirrup a read-only maintenance question before learning the MCP details. The immediate goal is modest: launch a real agent run and inspect either its answer or a useful error.

By the end of the tutorial we will connect the same run to **MCP tools**, a saved **trajectory**, and **evaluation**.

## 1. Locate AssetOpsBench and load configuration

Run Jupyter from the AssetOpsBench repository. The cell searches parent directories as a convenience. It reads `.env` without printing secret values.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request

from dotenv import load_dotenv

def find_repo(start=Path.cwd()):
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('AssetOpsBench repository not found. Start Jupyter from the repository.')

REPO = find_repo()
ENV_FILE = REPO / '.env'
if not ENV_FILE.exists():
    raise RuntimeError(f'Missing {ENV_FILE}. Complete 00_environment_setup.ipynb first.')
load_dotenv(ENV_FILE, override=True)
assert shutil.which('uv'), 'uv is not installed. Use 00_environment_setup.ipynb.'
print('repository:', REPO)
print('environment source:', ENV_FILE)
print('uv:', shutil.which('uv'))

## 2. Select the configured model

Put `KDD_MODEL_ID` and the matching provider credentials in the repository `.env`, then restart the kernel. Supported routes are `tokenrouter/...`, `litellm_proxy/...`, and `watsonx/...`.

In [ ]:
MODEL_ID = (os.getenv('KDD_MODEL_ID') or '').strip()
if not MODEL_ID:
    raise RuntimeError('Set KDD_MODEL_ID in .env, save it, and restart the kernel.')

credential_map = {
    'tokenrouter/': ['TOKENROUTER_API_KEY', 'TOKENROUTER_BASE_URL'],
    'litellm_proxy/': ['LITELLM_API_KEY', 'LITELLM_BASE_URL'],
    'watsonx/': ['WATSONX_APIKEY', 'WATSONX_PROJECT_ID'],
}
prefix = next((p for p in credential_map if MODEL_ID.startswith(p)), None)
if prefix is None:
    raise RuntimeError(f'Unsupported KDD_MODEL_ID route: {MODEL_ID}')
missing = [name for name in credential_map[prefix] if not os.getenv(name)]
print('agent: Stirrup')
print('model:', MODEL_ID)
print('credentials:', 'ready' if not missing else 'missing ' + ', '.join(missing))

def check_router_model_access(model_id, route_prefix, timeout=30):
    # TokenRouter and LiteLLM proxy both expose an OpenAI-compatible endpoint.
    if route_prefix == 'tokenrouter/':
        base_url = os.getenv('TOKENROUTER_BASE_URL', '').rstrip('/')
        api_key = os.getenv('TOKENROUTER_API_KEY', '')
    elif route_prefix == 'litellm_proxy/':
        base_url = os.getenv('LITELLM_BASE_URL', '').rstrip('/')
        api_key = os.getenv('LITELLM_API_KEY', '')
    else:
        return None, 'WatsonX access is checked by Stirrup when the run starts.'
    if not base_url or not api_key:
        return False, 'required router URL or API key is missing'
    payload = json.dumps({
        'model': model_id.split('/', 1)[1],
        'messages': [{'role': 'user', 'content': 'Reply with only OK.'}],
        'max_tokens': 8,
        'stream': False,
    }).encode('utf-8')
    request = urllib.request.Request(
        base_url + '/chat/completions',
        data=payload,
        headers={'Authorization': 'Bearer ' + api_key, 'Content-Type': 'application/json'},
        method='POST',
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            result = json.loads(response.read().decode('utf-8'))
        content = result.get('choices', [{}])[0].get('message', {}).get('content', '')
        return True, content.strip() or 'response received'
    except urllib.error.HTTPError as exc:
        detail = exc.reason
        try:
            error_payload = json.loads(exc.read().decode('utf-8'))
            detail = error_payload.get('error', {}).get('message') or error_payload.get('message') or detail
        except Exception:
            pass
        return False, f'HTTP {exc.code}: {detail}'
    except Exception as exc:
        return False, f'{type(exc).__name__}: {exc}'

if missing:
    MODEL_ACCESSIBLE, MODEL_ACCESS_MESSAGE = False, 'credentials are incomplete'
else:
    MODEL_ACCESSIBLE, MODEL_ACCESS_MESSAGE = check_router_model_access(MODEL_ID, prefix)
print('model access:', 'confirmed' if MODEL_ACCESSIBLE else ('deferred' if MODEL_ACCESSIBLE is None else 'failed'))
print('model response:', MODEL_ACCESS_MESSAGE)
if MODEL_ACCESSIBLE is False:
    raise RuntimeError('The configured model preflight failed: ' + MODEL_ACCESS_MESSAGE)

## 3. First agent query with `plan-execute`

The repository quick start uses this terminal command:

```bash
uv run plan-execute "What sensors are on Chiller 6?"
```

That exact command uses the CLI's default model. In this notebook, `--model-id` always receives the single `KDD_MODEL_ID` loaded from the repository `.env`. The cell retries transient HTTP 429/503 provider responses and saves full logs without filling the notebook with a traceback. The exercise is successful if it returns an answer, plan, or actionable error.

In [ ]:
PLAN_EXECUTE_QUESTION = 'What sensors are on Chiller 6?'
PLAN_EXECUTE_ACCESSIBLE = MODEL_ACCESSIBLE
PLAN_EXECUTE_ACCESS_MESSAGE = MODEL_ACCESS_MESSAGE
PLAN_EXECUTE_TIMEOUT_SECONDS = int(os.getenv('KDD_PLAN_EXECUTE_TIMEOUT_SECONDS', '180'))
PLAN_EXECUTE_ATTEMPTS = int(os.getenv('KDD_PLAN_EXECUTE_ATTEMPTS', '2'))
plan_execute_cmd = [
    'uv', 'run', '--directory', str(REPO), 'plan-execute',
    '--model-id', MODEL_ID,
    '--show-plan', '--show-trajectory',
    PLAN_EXECUTE_QUESTION,
]
plan_execute_env = os.environ.copy()
plan_execute_env.pop('VIRTUAL_ENV', None)
plan_log_dir = REPO / 'artifacts' / 'kdd_tutorial' / 'logs'
plan_log_dir.mkdir(parents=True, exist_ok=True)
plan_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
plan_stdout_path = plan_log_dir / f'kdd-plan-execute-{plan_stamp}.stdout.log'
plan_stderr_path = plan_log_dir / f'kdd-plan-execute-{plan_stamp}.stderr.log'
print('model from KDD_MODEL_ID:', MODEL_ID)
print('model access:', 'confirmed' if PLAN_EXECUTE_ACCESSIBLE else ('deferred' if PLAN_EXECUTE_ACCESSIBLE is None else 'failed'))
print('model response:', PLAN_EXECUTE_ACCESS_MESSAGE)
print('Running:', ' '.join(plan_execute_cmd[:-1]), repr(PLAN_EXECUTE_QUESTION))

plan_execute_result = None
plan_execute_timed_out = False
stdout_text = ''
stderr_text = ''
attempts = range(1, PLAN_EXECUTE_ATTEMPTS + 1) if PLAN_EXECUTE_ACCESSIBLE is not False else []
for attempt in attempts:
    try:
        plan_execute_result = subprocess.run(
            plan_execute_cmd,
            cwd=REPO,
            env=plan_execute_env,
            text=True,
            capture_output=True,
            timeout=PLAN_EXECUTE_TIMEOUT_SECONDS,
            check=False,
        )
    except subprocess.TimeoutExpired as exc:
        stdout_text = exc.stdout or ''
        stderr_text = exc.stderr or ''
        if isinstance(stdout_text, bytes):
            stdout_text = stdout_text.decode(errors='replace')
        if isinstance(stderr_text, bytes):
            stderr_text = stderr_text.decode(errors='replace')
        plan_execute_timed_out = True
        break

    stdout_text = plan_execute_result.stdout or ''
    stderr_text = plan_execute_result.stderr or ''
    combined = stdout_text + '\n' + stderr_text
    transient = any(marker in combined for marker in (
        'cache_only_cold', 'cache-only admission', 'Error code: 503',
        'ServiceUnavailable', 'Error code: 429', 'RateLimitError',
    ))
    if plan_execute_result.returncode == 0 or not transient:
        break
    if attempt < PLAN_EXECUTE_ATTEMPTS:
        print(f'Transient provider rejection on attempt {attempt}; retrying in 3 seconds...')
        time.sleep(3)

plan_stdout_path.write_text(stdout_text, encoding='utf-8')
plan_stderr_path.write_text(stderr_text, encoding='utf-8')
if PLAN_EXECUTE_ACCESSIBLE is False:
    print('plan-execute skipped because the selected model failed its access preflight.')
    print('Check the credentials/quota for KDD_MODEL_ID in the repository .env.')
elif plan_execute_timed_out:
    print(f'plan-execute exceeded {PLAN_EXECUTE_TIMEOUT_SECONDS}s.')
elif plan_execute_result.returncode == 0:
    print(stdout_text)
    print('plan-execute exit code: 0')
else:
    print('plan-execute exit code:', plan_execute_result.returncode)
    if transient:
        print('The model provider remained cold, overloaded, or rate-limited after retries.')
        print('This is not a CouchDB or MCP failure. Check KDD_MODEL_ID provider quota,')
        print('restart the kernel after any .env change, and rerun this cell.')
    else:
        print('The command failed. Inspect the saved stderr log for the full diagnostic.')
print('stdout log:', plan_stdout_path)
print('stderr log:', plan_stderr_path)
print('You may continue to the bounded Stirrup example below.')

## 4. Choose a Stirrup question

Start with `iot_sensors`. If it works, try `tsfm_lstm`. The other two questions demonstrate work-order retrieval and multi-server orchestration. All prompts are read-only.

In [ ]:
QUESTIONS = {
    'iot_sensors': 'What sensors are on Chiller 6?',
    'tsfm_lstm': 'Is LSTM model supported in TSFM?',
    'workorder': 'Get the work order of equipment CWC04013 for year 2017.',
    'multi_server': (
        'What is the current date and time? Also list assets at site MAIN. '
        'Also get the sensor list and failure mode list for any of the chiller at site MAIN.'
    ),
}
AGENT_QUESTIONS = {
    'iot_sensors': (
        'Call iot__installed_sensors exactly once with site_name MAIN and asset_id Chiller 6. '
        'Do not call sites, measured_sensors, or any other domain tool. Summarize the returned '
        'installed sensor names, then immediately call finish with that summary as the answer.'
    ),
    'tsfm_lstm': (
        'Use only the minimum read-only TSFM catalog tool calls needed to determine whether an '
        'LSTM model is supported. Then immediately call finish with a concise grounded answer.'
    ),
    'workorder': QUESTIONS['workorder'],
    'multi_server': QUESTIONS['multi_server'],
}
QUERY_NAME = 'iot_sensors'
DISPLAY_QUESTION = QUESTIONS[QUERY_NAME]
QUESTION = AGENT_QUESTIONS[QUERY_NAME]
print('Participant question:', DISPLAY_QUESTION)
print('Bounded agent instruction:', QUESTION)

## 5. Run Stirrup

`--no-code` keeps this demonstration on the MCP tools-only path. The command saves logs and requests a JSON trajectory. A first run may take longer while the environment starts.

In [ ]:
if missing:
    raise RuntimeError('Configure credentials and restart the kernel: ' + ', '.join(missing))

ARTIFACTS = REPO / 'artifacts' / 'kdd_tutorial'
TRACE_DIR = ARTIFACTS / 'trajectories'
LOG_DIR = ARTIFACTS / 'logs'
TRACE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ID = f'kdd-hello-{QUERY_NAME}-{stamp}'
SCENARIO_ID = f'kdd-hello-{QUERY_NAME}'
trajectory_path = TRACE_DIR / f'{RUN_ID}.json'
stdout_path = LOG_DIR / f'{RUN_ID}.stdout.log'
stderr_path = LOG_DIR / f'{RUN_ID}.stderr.log'

cmd = [
    'uv', 'run', '--directory', str(REPO), 'stirrup-agent',
    '--no-code', '--json', '--max-turns', '3',
    '--model-id', MODEL_ID,
    '--run-id', RUN_ID,
    '--scenario-id', SCENARIO_ID,
    QUESTION,
]
env = os.environ.copy()
env['AGENT_TRAJECTORY_DIR'] = str(TRACE_DIR)
timeout_seconds = int(os.getenv('KDD_AGENT_TIMEOUT_SECONDS', '120'))

try:
    completed = subprocess.run(
        cmd, cwd=REPO, env=env, text=True, capture_output=True, timeout=timeout_seconds
    )
    stdout_text = completed.stdout or ''
    stderr_text = completed.stderr or ''
    returncode = completed.returncode
except subprocess.TimeoutExpired as exc:
    stdout_text = exc.stdout or ''
    stderr_text = exc.stderr or ''
    if isinstance(stdout_text, bytes):
        stdout_text = stdout_text.decode(errors='replace')
    if isinstance(stderr_text, bytes):
        stderr_text = stderr_text.decode(errors='replace')
    returncode = None

stdout_path.write_text(stdout_text, encoding='utf-8')
stderr_path.write_text(stderr_text, encoding='utf-8')
print('exit code:', returncode if returncode is not None else 'timeout')
print('stdout log:', stdout_path)
print('stderr log:', stderr_path)
print('trajectory:', trajectory_path if trajectory_path.exists() else 'not persisted')
if returncode is None:
    print('DIAGNOSIS: the model preflight succeeded, but the multi-turn agent run stalled.')
    print('This normally means the selected model did not complete a later tool/finish turn.')
    print('Use a configured model with reliable native tool calling, then rerun this cell.')

## 6. Inspect the result

Do not worry yet about every field. Look for the final answer and whether the trajectory contains an executed tool call with an output.

In [ ]:
if trajectory_path.exists():
    trajectory = json.loads(trajectory_path.read_text(encoding='utf-8'))
    print('answer:', trajectory.get('answer'))
    calls = [
        call
        for turn in trajectory.get('trajectory', {}).get('turns', [])
        for call in turn.get('tool_calls', [])
    ]
    print('executed tools:', [call.get('name') for call in calls])
    display(trajectory)
else:
    print('No saved trajectory. Inspect the logs below before rerunning.')
    print('Last stdout lines:')
    print('\n'.join(stdout_text.splitlines()[-20:]))
    print('Last stderr lines:')
    print('\n'.join(stderr_text.splitlines()[-20:]))

## 7. What just happened?

The **host/agent** received the question. Stirrup and the model chose a capability. An **MCP client** sent a structured request to an AssetOpsBench **MCP server**. The server read a domain source or catalog and returned evidence. The agent then formed an answer, while the trajectory recorded the process.

In Hour 2 we call IoT and TSFM tools directly so these boundaries become visible. In Hour 3 we treat the question as a scenario and evaluate both the final answer and the trajectory.

## Optional second run

Change `QUERY_NAME` to `tsfm_lstm`, rerun from **Choose a question**, and compare the executed server/tool. Use `workorder` only after CouchDB data is loaded; use `multi_server` after Utilities, IoT, and FMSR are healthy.